# 05 — KIBA baseline experiment

## Goal

Configure one reproducible KIBA GCN/GAT/GIN + protein CNN run. The target is the source integrated KIBA score—not pKd—and remains separate from Davis.

**Execution policy:** `RUN_TRAINING = False` by default.


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import subprocess
import sys


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project checkout.")


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(
            f"Required v2 API {module_name}.{attribute} is unavailable. "
            "Update v2-refactor and rerun the tests before enabling this cell."
        )
    return getattr(module, attribute)


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if BRANCH != "v2-refactor":
    raise RuntimeError(f"Expected v2-refactor, found {BRANCH!r}. v1-final is read-only.")

print(f"Project: {PROJECT_ROOT}")
print(f"Branch:  {BRANCH}")


In [ ]:
from config import ExperimentConfig, SPLITS_DIR, processed_pairs_path

EXPERIMENT = ExperimentConfig(
    dataset="kiba",
    split="random",      # random | cold_drug | cold_target
    model="gcn",         # gcn | gat | gin
    seed=42,
    batch_size=32,
    epochs=30,
    learning_rate=1e-3,
    weight_decay=1e-5,
)
MANIFEST_PATH = SPLITS_DIR / "manifests" / (
    f"{EXPERIMENT.dataset}_{EXPERIMENT.split}_seed{EXPERIMENT.seed}_manifest.csv"
)
RUN_TRAINING = False

print(json.dumps(EXPERIMENT.to_dict(), indent=2))
print("Pairs:", processed_pairs_path("kiba"))
print("Manifest:", MANIFEST_PATH)


## Steps


In [ ]:
if RUN_TRAINING:
    run_baseline_experiment = require_api(
        "src.training", "run_baseline_experiment"
    )
    experiment_record = run_baseline_experiment(
        config=EXPERIMENT,
        split_manifest_path=MANIFEST_PATH,
    )
    print(json.dumps(experiment_record, indent=2, default=str))
else:
    print("Training skipped: RUN_TRAINING is False.")


## Checks

Confirm that all records and axes say KIBA score, checkpoint selection uses validation MSE, and the final test set is accessed only once after selection.

## Next steps

Complete the matched KIBA grid one run at a time, then use notebook 06 for cross-dataset baseline selection.
